## **Project Brief**

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS toast_proj;

CREATE OR REPLACE TABLE toast_proj.project_brief AS
SELECT
  'Toast Product Analyst' AS target_role,
  'PM, Online Ordering'  AS stakeholder,
  'Improve digital ordering adoption and conversion' AS decision,
  'Why did digital ordering conversion drop MoM, and which segments drive the change?' AS core_question,
  'Weekly' AS cadence,
  current_timestamp() AS created_at;


## **Requirements  (Metrics and Dimensions)**
SCAN

In [0]:
%sql
CREATE OR REPLACE TABLE toast_proj.requirements AS
SELECT * FROM VALUES
  ('metric',    'conversion', 'Order conversion rate from menu_view → checkout → order_complete'),
  ('dimension', 'time',       'week / month'),
  ('dimension', 'segment',    'restaurant_size, region, new vs existing'),
  ('dimension', 'channel',    'web vs app'),
  ('deliverable','tableau',   'Exec summary + drivers + segment deep dive + experiment readout');


Metrics Dictionary

In [0]:
%sql
CREATE OR REPLACE TABLE toast_proj.metric_definitions AS
SELECT * FROM VALUES
  ('north_star', 'digital_order_conversion', 'orders_completed / sessions_with_menu_view', 'Weekly'),
  ('driver', 'menu_to_checkout_cvr', 'sessions_checkout / sessions_menu_view', 'Weekly'),
  ('driver', 'checkout_to_order_cvr', 'orders_completed / sessions_checkout', 'Weekly'),
  ('driver', 'aov', 'total_revenue / orders_completed', 'Weekly'),
  ('driver', 'adoption_rate', 'restaurants_with_DO_enabled / total_restaurants', 'Monthly');


Data Quality Rules

In [0]:
%sql
CREATE OR REPLACE TABLE toast_proj.data_quality_rules AS
SELECT * FROM VALUES
  ('events', 'event_ts must not be null', 'event_ts IS NOT NULL'),
  ('events', 'event_name must be valid',  "event_name IN ('menu_view','add_to_cart','checkout_start','order_complete')"),
  ('orders', 'order_total must be > 0',   'order_total > 0');


This project uses a synthetic dataset designed to replicate Toast-like product event data, including realistic funnel drop-offs, volume, and behavioral patterns

## Bronze: Raw Tables - Synthetic Raw Data

Restaurants table (dimension)

In [0]:
%sql
CREATE OR REPLACE TABLE toast_proj.bronze_restaurants AS
SELECT
  id AS restaurant_id,
  element_at(array('NE','SE','MW','W','SW'), cast(rand()*5 as int)+1) AS region,
  element_at(array('SMB','MID','ENT'), cast(rand()*3 as int)+1) AS size_tier,
  date_add('2023-01-01', cast(rand()*700 as int)) AS onboard_date,
  case when rand() < 0.65 then true else false end AS do_enabled
FROM range(1, 2001);


Sessions table (fact-like grain: 1 row per session)

In [0]:
%sql
CREATE OR REPLACE TABLE toast_proj.bronze_sessions AS
SELECT
  uuid() AS session_id,
  cast(rand()*2000 as int)+1 AS restaurant_id,
  timestampadd(MINUTE, cast(rand()*60*24*365 as int), timestamp('2024-01-01')) AS session_ts,
  element_at(array('web','app'), cast(rand()*2 as int)+1) AS platform
FROM range(1, 250001);  -- 250K sessions


Events table (fact: 1 row per event)

In [0]:
%sql
CREATE OR REPLACE TABLE toast_proj.bronze_events AS
WITH base AS (
  SELECT session_id, restaurant_id, session_ts
  FROM toast_proj.bronze_sessions
),
steps AS (
  SELECT session_id, restaurant_id, session_ts, 'menu_view' AS event_name, 1 AS step, 1.00 AS p FROM base
  UNION ALL
  SELECT session_id, restaurant_id, session_ts, 'add_to_cart', 2, 0.55 FROM base
  UNION ALL
  SELECT session_id, restaurant_id, session_ts, 'checkout_start', 3, 0.35 FROM base
  UNION ALL
  SELECT session_id, restaurant_id, session_ts, 'order_complete', 4, 0.22 FROM base
)
SELECT
  session_id,
  restaurant_id,
  event_name,
  timestampadd(SECOND, step * cast(rand()*120 as int), session_ts) AS event_ts
FROM steps
WHERE rand() < p;


Orders table

In [0]:
%sql
CREATE OR REPLACE TABLE toast_proj.bronze_orders AS
SELECT
  e.session_id,
  e.restaurant_id,
  e.event_ts AS order_ts,
  round(10 + rand()*80, 2) AS order_total
FROM toast_proj.bronze_events e
WHERE e.event_name = 'order_complete'
AND rand() > 0.01; -- small missingness


## Silver: Clean and Standardize 

Clean Events - formatted the event_ts as a timestamp, removed where event_ts is null, and only showing events that in the order funnel 

In [0]:
%sql
CREATE OR REPLACE TABLE toast_proj.silver_events AS
SELECT
  session_id,
  restaurant_id,
  event_name,
  cast(event_ts as timestamp) AS event_ts
FROM toast_proj.bronze_events
WHERE event_ts IS NOT NULL
  AND event_name IN ('menu_view','add_to_cart','checkout_start','order_complete');


Clean Orders - formatted the event_ts as a timestamp and removed orders where the order_total is 0

In [0]:
%sql
CREATE OR REPLACE TABLE toast_proj.silver_orders AS
SELECT
  session_id,
  restaurant_id,
  cast(order_ts as timestamp) AS order_ts,
  order_total
FROM toast_proj.bronze_orders
WHERE order_total > 0;


## Gold: Analytics layer (star schema and metric tables) - The curated data

Dim/Describers Table: restaurants

In [0]:
%sql
CREATE OR REPLACE TABLE toast_proj.dim_restaurants AS
SELECT * FROM toast_proj.bronze_restaurants;


Fact: funnel metrics by restaurant + week

In [0]:
%sql
CREATE OR REPLACE TABLE toast_proj.fact_funnel_weekly AS
WITH e AS (
  SELECT
    restaurant_id,
    date_trunc('week', event_ts) AS week,
    session_id,
    max(case when event_name='menu_view' then 1 else 0 end) AS menu_view,
    max(case when event_name='add_to_cart' then 1 else 0 end) AS add_to_cart,
    max(case when event_name='checkout_start' then 1 else 0 end) AS checkout_start,
    max(case when event_name='order_complete' then 1 else 0 end) AS order_complete
  FROM toast_proj.silver_events
  GROUP BY restaurant_id, date_trunc('week', event_ts), session_id
),
--Aggregations
agg AS (
  SELECT
    restaurant_id,
    week,
    count(*) AS sessions,
    sum(menu_view) AS sessions_menu_view,
    sum(add_to_cart) AS sessions_add_to_cart,
    sum(checkout_start) AS sessions_checkout,
    sum(order_complete) AS orders
  FROM e
  GROUP BY restaurant_id, week
)
SELECT
  restaurant_id,
  week,
  sessions,
  sessions_menu_view,
  sessions_add_to_cart,
  sessions_checkout,
  orders,
  orders / nullif(sessions_menu_view,0) AS digital_order_conversion,
  sessions_add_to_cart / nullif(sessions_menu_view,0) AS menu_to_cart_cvr,
  sessions_checkout / nullif(sessions_menu_view,0) AS menu_to_checkout_cvr,
  orders / nullif(sessions_checkout,0) AS checkout_to_order_cvr
FROM agg;


Fact: Weekly Revenue

In [0]:
%sql
CREATE OR REPLACE TABLE toast_proj.fact_revenue_weekly AS
SELECT
  restaurant_id,
  date_trunc('week', order_ts) AS week,
  count(*) AS orders,
  sum(order_total) AS revenue,
  avg(order_total) AS aov
FROM toast_proj.silver_orders
GROUP BY restaurant_id, date_trunc('week', order_ts);


**Initial EDA**
- What changed?
- Why did it change?
- Who / what is driving it?
- What should we do next?

Why did digital _ordering conversion_ drop _month over month_, and which _segments drove the change_?


In [0]:
%sql
SELECT
  week,
  AVG(digital_order_conversion) AS avg_conversion
FROM toast_proj.fact_funnel_weekly
GROUP BY week
ORDER BY week;


Databricks visualization. Run in Databricks to view.

**Contextual Insight:** 
After looking doing a broad trend analysis, week to week over a years time I noticed... 

**Average conversion** is hovering roughly **~0.21–0.23**

Week-to-week movement exists, but:
- No sustained upward trend
- no sustained downward trend
- no clear structural break
- One or two single-week dips, but they recover quickly

At an aggregate level, conversion looks stable with normal volatility

### Funnel Analysis
The flow of checkout is:
**menu_view → checkout → order_complete**

By splitting the process we'll be able to analyze where the decline in order conversion is prominent during the customer's journey


In [0]:
%sql
SELECT
  week,
  AVG(menu_to_checkout_cvr) AS menu_to_checkout,
  AVG(checkout_to_order_cvr) AS checkout_to_order
FROM toast_proj.fact_funnel_weekly
GROUP BY week
ORDER BY week;


Databricks visualization. Run in Databricks to view.

**Directional Insight:** Most of the decline is driven by checkout → order conversion, hinting that there may be a sign of purchase friction amongst customers who moved to checkout but didn't complete their order 

**Segment Analysis:** Where is the most change being driven from?



**By Restaurant Size**

In [0]:
%sql
SELECT
  d.size_tier,
  week,
  AVG(f.digital_order_conversion) AS conversion
FROM toast_proj.fact_funnel_weekly f
JOIN toast_proj.dim_restaurants d
  ON f.restaurant_id = d.restaurant_id
GROUP BY d.size_tier, week
ORDER BY week, d.size_tier;


Databricks visualization. Run in Databricks to view.

There is no size-tier–specific conversion regression.

ENT, MD, SMB all move together

No size tier shows:
- a sustained downward slope
- widening divergence over time

Volatility exists, but:
- it is symmetric
- it mean-reverts

Interpretation: There is no size-tier–specific conversion regression

**By Region**

In [0]:
%sql
SELECT
  d.region,
  week,
  AVG(f.digital_order_conversion) AS conversion
FROM toast_proj.fact_funnel_weekly f
JOIN toast_proj.dim_restaurants d
  ON f.restaurant_id = d.restaurant_id
GROUP BY d.region, week
ORDER BY week, d.region;


Databricks visualization. Run in Databricks to view.

All regions oscillate around ~0.21–0.24

No region:
- breaks away structurally
- shows prolonged underperformance

Short-term dips occur, but:
- they recover quickly
- they appear across regions

Interpretation:
There is no regionally isolated conversion issue.

**Initial EDA Relection**: At both the restaurant-size and regional levels, digital ordering conversion remains stable with no sustained divergence. This suggests that any observed aggregate volatility is not driven by a specific customer segment or geography

## DRIVER DECOMP.

Is menu → checkout down but checkout → order up?

Did order count drop while AOV rose?

Are retries masking failures?

### Funnel Decomp

**Goal:**
Break digital ordering conversion into its component steps to see whether:
- one step regressed
- another step improved
- and if the net effect looks “flat” at the top line

_Funnel Overview_
- Session / Menu View
- Sessions Menu View / Checkout Start
- Orders / Order Completed

Total Conversion = (menu → checkout) × (checkout → order)


**Funnel Count**

In [0]:
%sql
--funnel count
SELECT
  week,
  SUM(sessions)               AS sessions,
  SUM(sessions_menu_view)     AS sessions_with_menu,
  SUM(orders)                 AS orders
FROM toast_proj.fact_funnel_weekly
GROUP BY week
ORDER BY week;


**Funnel Decomposition**

In [0]:
%sql
SELECT
  week,
  SUM(sessions_menu_view) * 1.0 / SUM(sessions)           AS session_to_menu_cvr,
  SUM(orders) * 1.0 / SUM(sessions_menu_view)             AS menu_to_order_cvr,
  SUM(orders) * 1.0 / SUM(sessions)                        AS total_session_to_order_cvr
FROM toast_proj.fact_funnel_weekly
GROUP BY week
ORDER BY week;


In [0]:
%sql
SELECT
  week,
  AVG(menu_to_cart_cvr) AS reported_menu_to_cart,
  SUM(orders) * 1.0 / SUM(sessions_menu_view) AS calculated_menu_to_order
FROM toast_proj.fact_funnel_weekly
GROUP BY week
ORDER BY week;


Funnel decomposition was adapted to session-based metrics available in production tables, reflecting realistic product analytics constraints

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW funnel_rates_weekly AS
SELECT
  week,
  SUM(sessions_menu_view) * 1.0 / SUM(sessions)       AS session_to_menu_cvr,
  SUM(orders) * 1.0 / SUM(sessions_menu_view)         AS menu_to_order_cvr,
  SUM(orders) * 1.0 / SUM(sessions)                   AS total_cvr
FROM toast_proj.fact_funnel_weekly
GROUP BY week;


In [0]:
%sql
SELECT * FROM funnel_rates_weekly ORDER BY week;


Databricks visualization. Run in Databricks to view.

Findings - Nearly all sessions reach the menu, so the primary source of drop-off occurs between menu view and order completion. As a result, total conversion (green line) closely tracks menu-to-order conversion (yellow line).

Closer look

In [0]:
%sql
SELECT
  week,
  SUM(sessions) AS sessions,
  SUM(sessions_menu_view) AS menu_views,
  SUM(orders) AS orders,

  SUM(sessions_menu_view) * 1.0 / NULLIF(SUM(sessions), 0) AS session_to_menu_cvr,
  SUM(orders) * 1.0 / NULLIF(SUM(sessions_menu_view), 0)   AS menu_to_order_cvr,
  SUM(orders) * 1.0 / NULLIF(SUM(sessions), 0)             AS total_cvr
FROM toast_proj.fact_funnel_weekly f
GROUP BY week
ORDER BY week;



Databricks visualization. Run in Databricks to view.

By Volume

In [0]:
%sql
SELECT
  week,
  SUM(sessions) AS sessions,
  SUM(sessions_menu_view) AS menu_sessions,
  SUM(orders) AS orders
FROM toast_proj.fact_funnel_weekly
GROUP BY week
ORDER BY week;


Databricks visualization. Run in Databricks to view.

Digital ordering drop-off occurs almost entirely after menu view, with total conversion closely tracking menu-to-order behavior

### Final Conclusion: 

Funnel decomposition shows that nearly all sessions reach the menu, with a strong drop-off  downstream. As a result, total conversion closely mirrors menu-to-order conversion, indicating that there should be more of a focus on post-menu interactions rather than discovery.


**Funnel Decomposition & Key Insights
Objective**
Identify where user drop-off occurs within the digital ordering funnel and determine whether aggregate conversion stability masks underlying behavioral changes.

**Funnel Definition**
The digital ordering funnel was evaluated at the session level using the following steps:

- Session Started
- _Session with Menu View_
- _Order Completed_

Due to data availability, checkout-start events were not explicitly modeled. Analysis focuses on the transition from menu view to order completion.

**Key Findings**

Aggregate digital ordering conversion is stable
- Weekly total session-to-order conversion remained within a narrow historical range, showing no sustained upward or downward trend.
- This indicates the absence of a platform-wide regression in digital ordering performance.

Nearly all sessions reach the menu

- Session-to-menu conversion is consistently close to 100%, indicating that users rarely drop off before viewing the menu.
- As a result, early-funnel discovery is not a meaningful source of conversion loss from what was observed.

Drop-off is concentrated after menu view
- Menu-to-order conversion accounts for nearly all observed funnel attrition.
- Total session-to-order conversion closely tracks menu-to-order conversion over time, showing that downstream behavior — rather than traffic or discovery — is a major driver in overall performance.

Funnel behavior is consistent across segments
- Segment-level analysis by restaurant size and region shows similar funnel patterns, with no sustained divergence across cohorts.
- Suggests that experience of post-menu friction is broad rather than isolated to specific customer segments or geographies.

**Interpretation:**
The digital ordering funnel behaves as a single-step funnel beyond menu view. Improvements or regressions in total conversion are therefore driven primarily by **post-menu interaction**s, such as 
- pricing clarity
- checkout experience
- payment reliability.


**Implications for Product Focus**
Given that nearly all users reach the menu, optimization efforts should prioritize:
- Reducing friction between menu view and order completion
- Improving clarity and confidence during the ordering process
- Investigating post-menu UX elements that influence purchase commitment
- Early-funnel discovery improvements are unlikely to materially impact overall conversion.



**Data Limitations**
- Checkout-start events were not available in the analyzed schema.
- Funnel analysis was conducted using session-level aggregates rather than raw event logs.
- Findings reflect synthetic, representative data designed to simulate Toast-like product behavior.

## Final Insights 

### Decision
Focus product investment on improving post-menu ordering experience rather than early-funnel discovery.

### Primary Insight
Nearly all sessions reach the menu, with drop-off concentrated between menu view and order completion. Total conversion closely mirrors menu-to-order conversion.

### Supporting Evidence
- Stable aggregate conversion over time
- Session-to-menu conversion ~100%
- Consistent behavior across size tiers and regions

### Confidence & Limitations
High confidence in funnel location of drop-off; medium confidence in specific UX drivers due to lack of checkout-start instrumentation.


In [0]:
%sql
CREATE SCHEMA workspace.toast_proj_gold;

In [0]:
%sql
CREATE TABLE workspace.toast_proj_gold.funnel_weekly AS
SELECT
  week,
  SUM(sessions) AS sessions,
  SUM(sessions_menu_view) AS sessions_reaching_menu,
  SUM(orders) AS orders,

  SUM(sessions_menu_view) * 1.0 / NULLIF(SUM(sessions), 0) AS session_to_menu_cvr,
  SUM(orders) * 1.0 / NULLIF(SUM(sessions_menu_view), 0) AS menu_to_order_cvr,
  SUM(orders) * 1.0 / NULLIF(SUM(sessions), 0) AS total_cvr
FROM workspace.toast_proj.fact_funnel_weekly
GROUP BY week;

In [0]:
%sql
SELECT * FROM workspace.toast_proj_gold.restaurant_summary LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.toast_proj_gold.funnel_weekly_by_segment AS
SELECT
  f.week,
  d.size_tier,
  d.region,
  SUM(f.sessions) AS sessions,
  SUM(f.sessions_menu_view) AS sessions_reaching_menu,
  SUM(f.orders) AS orders,

  SUM(f.sessions_menu_view) * 1.0 / NULLIF(SUM(f.sessions), 0) AS session_to_menu_cvr,
  SUM(f.orders) * 1.0 / NULLIF(SUM(f.sessions_menu_view), 0)   AS menu_to_order_cvr,
  SUM(f.orders) * 1.0 / NULLIF(SUM(f.sessions), 0)             AS total_cvr

FROM workspace.toast_proj.fact_funnel_weekly f
JOIN workspace.toast_proj.dim_restaurants d
  ON f.restaurant_id = d.restaurant_id
GROUP BY f.week, d.size_tier, d.region;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.toast_proj_gold.restaurant_summary AS
SELECT
  f.restaurant_id,
  d.size_tier,
  d.region,
  SUM(f.sessions) AS sessions,
  SUM(f.sessions_menu_view) AS sessions_reaching_menu,
  SUM(f.orders) AS orders,

  SUM(f.orders) * 1.0 / NULLIF(SUM(f.sessions), 0) AS total_cvr,
  SUM(f.orders) * 1.0 / NULLIF(SUM(f.sessions_menu_view), 0) AS menu_to_order_cvr

FROM workspace.toast_proj.fact_funnel_weekly f
JOIN workspace.toast_proj.dim_restaurants d
  ON f.restaurant_id = d.restaurant_id
GROUP BY f.restaurant_id, d.size_tier, d.region;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.toast_proj_gold.metric_definitions (
  metric_name           STRING,
  metric_group          STRING,   -- Funnel / Revenue / Quality / Adoption etc.
  definition            STRING,
  numerator             STRING,
  denominator           STRING,
  formula               STRING,
  grain                 STRING,   -- e.g., week / restaurant-week / session
  source_table          STRING,   -- where metric is computed from
  filters_assumptions   STRING,   -- exclusions, time windows, etc.
  owner                 STRING,
  last_updated          DATE,
  notes                 STRING
)
USING DELTA;

INSERT INTO workspace.toast_proj_gold.metric_definitions
VALUES
-- Funnel metrics
(
 'session_to_menu_cvr',
 'Funnel',
 'Share of sessions that reached a menu view.',
 'SUM(sessions_menu_view)',
 'SUM(sessions)',
 'SUM(sessions_menu_view) / NULLIF(SUM(sessions),0)',
 'week',
 'workspace.toast_proj.fact_funnel_weekly',
 'Assumes sessions_menu_view represents sessions with at least one menu view.',
 'Kahiem',
 CURRENT_DATE(),
 'Near-100% indicates minimal early-funnel drop-off.'
),
(
 'menu_to_order_cvr',
 'Funnel',
 'Share of menu-reaching sessions that completed an order.',
 'SUM(orders)',
 'SUM(sessions_menu_view)',
 'SUM(orders) / NULLIF(SUM(sessions_menu_view),0)',
 'week',
 'workspace.toast_proj.fact_funnel_weekly',
 'Checkout-start not instrumented; proxy is menu session to completed order.',
 'Kahiem',
 CURRENT_DATE(),
 'Primary post-menu friction indicator.'
),
(
 'total_cvr',
 'Funnel',
 'Share of sessions that completed an order.',
 'SUM(orders)',
 'SUM(sessions)',
 'SUM(orders) / NULLIF(SUM(sessions),0)',
 'week',
 'workspace.toast_proj.fact_funnel_weekly',
 'Blends all funnel steps; mirrors menu_to_order when session_to_menu is ~1.0',
 'Kahiem',
 CURRENT_DATE(),
 'Topline conversion.'
),

-- Revenue metrics
(
 'revenue',
 'Revenue',
 'Total order revenue for the period.',
 'SUM(order_total)',
 NULL,
 'SUM(order_total)',
 'week',
 'workspace.toast_proj.fact_revenue_weekly',
 'Excludes orders with order_total = 0 (clean rule in Silver).',
 'Kahiem',
 CURRENT_DATE(),
 'Use with conversion to estimate lift impact.'
),
(
 'aov',
 'Revenue',
 'Average order value for the period.',
 'SUM(order_total)',
 'COUNT(DISTINCT order_id)',
 'SUM(order_total)/NULLIF(COUNT(DISTINCT order_id),0)',
 'week',
 'workspace.toast_proj.fact_revenue_weekly',
 'Order grain assumptions depend on revenue fact table design.',
 'Kahiem',
 CURRENT_DATE(),
 'Pairs with order_count to explain revenue changes.'
);





In [0]:
%sql
SELECT * FROM workspace.toast_proj_gold.metric_definitions
ORDER BY metric_group, metric_name;

In [0]:
%sql
SELECT * FROM workspace.toast_proj_gold.restaurant_summary;
